In [0]:

from __future__ import annotations

import argparse
import json
import os

from inference_artifacts import (
    build_request_from_env,
    build_spark_session,
    build_default_run_id,
    build_default_run_ts,
    get_base_path,
    load_and_validate_champion_row_for_station,
    load_champion_station_ids,
    resolve_canonical_station_id,
    resolve_inference_run_context,
    to_serializable_dict,
    to_utc_hour,
    validate_canonical_station_against_champion_scope,
    write_step_artifact,
    build_live_station_snapshot,
)


def run_live_station_step(
    run_id: str,
    run_ts: str,
) -> tuple[dict, str]:
    base_path = get_base_path()

    request = build_request_from_env()
    request_ts = to_utc_hour(request.request_timestamp)

    live_station = build_live_station_snapshot(request)
    request.name = live_station.name

    resolver_result = resolve_canonical_station_id(base_path, live_station, request)
    canonical_station_id = str(resolver_result["canonical_station_id"])

    spark = build_spark_session()
    try:
        champion_station_ids = load_champion_station_ids(spark, base_path)
        validate_canonical_station_against_champion_scope(
            canonical_station_id=canonical_station_id,
            champion_station_ids=champion_station_ids,
        )
        champion_row = load_and_validate_champion_row_for_station(
            spark=spark,
            base_path=base_path,
            canonical_station_id=canonical_station_id,
        )
    finally:
        spark.stop()

    payload = {
        "run_id": run_id,
        "run_ts": run_ts,
        "request": {
            "station_id": request.station_id,
            "name": request.name,
            "lat": request.lat,
            "lon": request.lon,
        },
        "request_timestamp": request_ts.isoformat().replace("+00:00", "Z"),
        "live_station": to_serializable_dict(live_station),
        "canonical_station_id": canonical_station_id,
        "resolver_result": resolver_result,
        "champion_row": champion_row,
    }

    artifact_path = write_step_artifact(
        base_path=base_path,
        run_id=run_id,
        step_name="live_station",
        payload=payload,
        filename="live_station.json",
    )
    return payload, artifact_path


def main() -> None:
    parser = argparse.ArgumentParser(description="Inference Step 01: live station context")
    parser.add_argument("--run-id", default=None)
    parser.add_argument("--run-ts", default=None)
    parser.add_argument(
        "--request-json",
        default=None,
        help="Optional INFERENCE_REQUEST_JSON override",
    )
    args = parser.parse_args()

    if args.request_json and args.request_json.strip():
        os.environ["INFERENCE_REQUEST_JSON"] = args.request_json.strip()

    run_id, run_ts = resolve_inference_run_context(args.run_id, args.run_ts)

    payload, artifact_path = run_live_station_step(run_id=run_id, run_ts=run_ts)
    print(
        json.dumps(
            {
                "step": "live_station",
                "run_id": run_id,
                "run_ts": run_ts,
                "artifact_path": artifact_path,
                "station_id": payload["live_station"]["station_id"],
                "canonical_station_id": payload["canonical_station_id"],
            },
            indent=2,
        )
    )


In [0]:
def store_databricks_params_as_env():
    if os.getenv("DATABRICKS_RUNTIME_VERSION"):
        # 1. Retrieve all parameters (widgets) as a dictionary
        # This returns a dict of { "parameter_name": "parameter_value" }
        all_params = dbutils.widgets.getAll()

        # 2. Store them in os.environ
        for key, value in all_params.items():
            os.environ[key] = str(value)

        print(f"Stored {len(all_params)} parameters in environment variables.")

def build_default_run_id() -> str:
    env_run_id = os.environ.get("INFERENCE_RUN_ID")
    if env_run_id and env_run_id.strip():
        return env_run_id.strip()

    pipeline_run_id = os.environ.get("PIPELINE_RUN_ID")
    if pipeline_run_id and pipeline_run_id.strip():
        return pipeline_run_id.strip()

    job_run_id = os.environ.get("PIPELINE_JOB_RUN_ID")
    if job_run_id and job_run_id.strip():
        repair_count = os.environ.get("PIPELINE_REPAIR_COUNT", "0").strip() or "0"
        return f"job_{job_run_id.strip()}_repair_{repair_count}"

    now_utc = datetime.datetime.now(timezone.utc)
    return f"run_{now_utc.strftime('%Y%m%d%H%M%S')}_{uuid.uuid4().hex[:8]}"


def build_default_run_ts() -> str:
    env_run_ts = os.environ.get("INFERENCE_RUN_TS")
    if env_run_ts and env_run_ts.strip():
        return env_run_ts.strip()

    pipeline_run_ts = os.environ.get("PIPELINE_RUN_TS")
    if pipeline_run_ts and pipeline_run_ts.strip():
        return pipeline_run_ts.strip()

    return datetime.datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")

def is_job():
    try:
        # Access the internal notebook context
        context_json = dbutils.notebook.entry_point.getDbutils().notebook().getContext().toJson()
        context = json.loads(context_json)
        
        # Check for the existence of jobId in tags
        return "jobId" in context.get("tags", {})
    except:
        return False

import datetime
from datetime import timedelta, timezone
import uuid

if is_job():
    print("Running as a Databricks Job")
else:
    print("Running interactively (Manual)")
    now_utc = datetime.datetime.now(timezone.utc)
    dbutils.widgets.text("INFERENCE_REQUEST_JSON", '{"name":"du Mont-Royal / Clark","lat":45.51941,"lon":-73.58685,"station_id":"218"}')
    dbutils.widgets.text("INFERENCE_RUN_ID", "20260420080036_e9483314") #f"{now_utc.strftime('%Y%m%d%H%M%S')}_{uuid.uuid4().hex[:8]}")
    dbutils.widgets.text("INFERENCE_HORIZON_STEPS", "3")

In [0]:
store_databricks_params_as_env()

request_json = os.getenv("INFERENCE_REQUEST_JSON")
run_id = os.getenv("INFERENCE_RUN_ID")
horizon_steps = os.getenv("INFERENCE_HORIZON_STEPS")

print("args request json:", request_json)
print("args horizon_steps id:", horizon_steps)
print("args run_id:", run_id)

In [0]:
run_id, run_ts = resolve_inference_run_context(run_id, build_default_run_ts())
payload, artifact_path = run_live_station_step(run_id=run_id, run_ts=run_ts)
print(
    json.dumps(
        {
            "step": "live_station",
            "run_id": run_id,
            "run_ts": run_ts,
            "artifact_path": artifact_path,
            "station_id": payload["live_station"]["station_id"],
            "canonical_station_id": payload["canonical_station_id"],
        },
        indent=2,
    )
)
